In [0]:
import delta

class Ingestor:
    def __init__(self, spark, source_path, data_format, target_path, catalog, schemaname, tablename):
        self.spark = spark
        self.source_path = source_path
        self.data_format = data_format
        self.target_path = target_path
        self.catalog = catalog
        self.schemaname = schemaname
        self.tablename = tablename
    
    def load(self, source_path):
        df = (
            spark.read.format(self.data_format) \
                .load(source_path)
        )
        return df

    def save(self, df):
        (df.write
         .format("delta")
         .option("path", self.target_path)
         .option("overwriteSchema", "true")
         .mode("overwrite")
         .saveAsTable(f"{self.catalog}.{self.schemaname}.{self.tablename}")
         )
        return True
    
    def executeLoadAndSave(self, path):
        df = self.load(path)
        return self.save(df)
    
class IncrementalIngestor(Ingestor):
    def __init__(self, spark, source_path, data_format, target_path, catalog, schemaname, tablename, schema_location,checkpoint_location, id_field, timestamp_field):
        super().__init__(spark, source_path, data_format, target_path, catalog, schemaname, tablename)
        self.timestamp_field = timestamp_field
        self.id_field = id_field
        self.checkpoint_location = checkpoint_location
        self.schema_location = schema_location
        self.set_deltatable()
        

    def set_deltatable(self):
        table = f"{self.catalog}.{self.schemaname}.{self.tablename}"
        self.delta_table = delta.DeltaTable.forName(self.spark, table)

    def upsert(self, df):
        
        df.createOrReplaceGlobalTempView(f"view_{self.tablename}")

        if self.timestamp_field == "N/A":
            query = f"""
        SELECT * 
        from global_temp.view_{self.tablename}
        """

        else:
            query = f"""
            SELECT * 
            from global_temp.view_{self.tablename}
            QUALIFY ROW_NUMBER() OVER(PARTITION BY {self.id_field} ORDER BY {self.timestamp_field} DESC) = 1
            """
            
        query_incremental = self.spark.sql(query)

        print("executou upsert")

        (
            self.delta_table.alias("t") \
                .merge(
                    query_incremental.alias("s"),
                    f"s.{self.id_field} = t.{self.id_field}"
                ) \
                .whenMatchedUpdateAll() \
                .whenNotMatchedInsertAll() \
                .execute()
        )

    def load(self, source_path):
       
        df = (
            spark.readStream.format("cloudFiles") \
                .option("cloudFiles.Format", self.data_format) \
                .option('cloudFiles.inferColumnTypes', 'true')
                .option("cloudFiles.schemaLocation", self.schema_location) \
                .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
                .load(source_path)
        )

        print("executou load")

        return df
    
    def save(self, df):
        print("executou save")
        stream = (df.writeStream
         .format("delta")
         .option("checkpointLocation", self.checkpoint_location)
         .option("path", self.target_path)
         .foreachBatch(lambda df, batchId: self.upsert(df))
         .trigger(availableNow=True)
        )
        return stream.start()

    def executeLoadAndSave(self, path):
        print("executou load and save")
        df = self.load(path)
        return self.save(df)
